In [1]:
# 1. Imports and Setup
import pandas as pd
import numpy as np
import os
import glob
import itertools
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, HTML
import warnings

warnings.filterwarnings('ignore')
pd.options.mode.chained_assignment = None

display(HTML("<style>.container { width:100% !important; }</style>"))
plt.rcParams['figure.figsize'] = (24, 8)
sns.set_theme(style="darkgrid")

DATA_DIR = r'D:\0dot1_Aug_2016_master\data\mstock_mtf_daily_data'


In [2]:
# 2. Data Loading (Entire Market Dataset)
def load_all_data():
    all_data = []
    files = glob.glob(os.path.join(DATA_DIR, '*.csv'))[:50] # Limit to 50 stocks
    print(f"Found {len(files)} stock files. Loading...")
    
    for file_path in files:
        symbol = os.path.basename(file_path).replace('.csv', '')
        try:
            df = pd.read_csv(file_path)
            df['Date'] = pd.to_datetime(df['Date'])
            df = df.sort_values('Date').reset_index(drop=True)
            df['Symbol'] = symbol
            all_data.append(df)
        except Exception as e:
            pass
            
    print(f"Loaded {len(all_data)} stocks successfully.")
    return pd.concat(all_data, ignore_index=True)

data = load_all_data()


Found 50 stock files. Loading...


Loaded 50 stocks successfully.


In [3]:
# 3. Regimes & Forward Returns Calculation
LOOKBACKS = [13, 21, 34]
FORWARD_DAYS = [5, 8, 13]

def enrich_data(df):
    df = df.copy()
    
    # 1. Forward Returns
    for d in FORWARD_DAYS:
        df[f'Fwd_{d}d'] = df['Close'].shift(-d) / df['Close'] - 1
        
    # 2. Static ER & WMA Calculations
    for p in LOOKBACKS:
        change = abs(df['Close'] - df['Close'].shift(p))
        volatility = df['Close'].diff().abs().rolling(p).sum()
        df[f'ER_{p}'] = change / volatility
        
        # WMA Distance
        weights = np.arange(1, p + 1)
        wma = df['Close'].rolling(p).apply(lambda x: np.dot(x, weights) / weights.sum(), raw=True)
        df[f'WMA_Dist_{p}'] = (df['Close'] - wma) / wma
        
    return df

print("Pre-calculating ERs and Forward Returns for all stocks... (This may take a minute)")
_data_list = []
for sym, grp in data.groupby('Symbol'):
    _data_list.append(enrich_data(grp))
data = pd.concat(_data_list, ignore_index=True)
data = data.dropna(subset=[f'Fwd_{d}d' for d in FORWARD_DAYS] + [f'ER_{p}' for p in LOOKBACKS])
print("Data enrichment complete!")


Pre-calculating ERs and Forward Returns for all stocks... (This may take a minute)


Data enrichment complete!


In [4]:
# 4. Walk-Forward Grid Profiler with Risk Avoidance
TRAIN_END_DATE = '2023-01-01'
ENTRY_THRESHOLDS = [0.4, 0.5, 0.6]
WMA_AVOID_THRESHOLDS = [0.03, 0.05, 0.08] # Max allowed distance above WMA

train_data = data[data['Date'] < TRAIN_END_DATE]
test_data = data[data['Date'] >= TRAIN_END_DATE]

leaderboard = []
best_score = -9999
best_params = {}

print("Running Grid Search on Training Data and Blind Testing on Unseen 2023+ Data...")

for p in LOOKBACKS:
    for er_thresh in ENTRY_THRESHOLDS:
        for wma_max in WMA_AVOID_THRESHOLDS:
            # Filter training days: ER must be HIGH, but WMA Distance must be LOW (not overextended)
            train_trades = train_data[(train_data[f'ER_{p}'] > er_thresh) & (train_data[f'WMA_Dist_{p}'] < wma_max)]
            
            if len(train_trades) < 50: 
                continue # Skip if it rarely happens across the whole market
                
            for d in FORWARD_DAYS:
                # 1. Evaluate on Training Data
                train_winrate = (train_trades[f'Fwd_{d}d'] > 0).mean()
                if train_winrate < 0.45: 
                    continue # Skip if it didn't even work in the past
                    
                # 2. Evaluate on Unseen Testing Data
                test_trades = test_data[(test_data[f'ER_{p}'] > er_thresh) & (test_data[f'WMA_Dist_{p}'] < wma_max)]
                
                if len(test_trades) < 10: 
                    continue
                    
                test_winrate = (test_trades[f'Fwd_{d}d'] > 0).mean()
                test_avg_ret = test_trades[f'Fwd_{d}d'].mean()
                
                # Expectancy score penalized by low trade count
                fitness = test_winrate * test_avg_ret * (len(test_trades) ** 0.5)
                
                leaderboard.append({
                    'Lookback': p,
                    'ER_Entry': f"> {er_thresh}",
                    'WMA_Avoid': f"< {wma_max}",
                    'Hold_Days': d,
                    'Test_Trades': len(test_trades),
                    'Test_WinRate': test_winrate * 100,
                    'Test_AvgReturn': test_avg_ret * 100,
                    'Fitness': fitness
                })

leaderboard_df = pd.DataFrame(leaderboard).sort_values('Fitness', ascending=False).reset_index(drop=True)
display(HTML("<h3>Full Market Leaderboard (Tested on Unseen 2023+ Data)</h3>"))
display(leaderboard_df.head(20))


Running Grid Search on Training Data and Blind Testing on Unseen 2023+ Data...


,Lookback,ER_Entry,WMA_Avoid,Hold_Days,Test_Trades,Test_WinRate,Test_AvgReturn,Fitness
0,34,> 0.6,< 0.03,13,134,68.656716,11.763476,0.934913
1,34,> 0.6,< 0.05,13,136,67.647059,11.412681,0.900339
2,34,> 0.6,< 0.08,13,154,66.233766,10.179746,0.836713
3,34,> 0.5,< 0.03,13,409,56.479218,5.153051,0.588593
4,34,> 0.5,< 0.05,13,442,57.466063,4.825321,0.582974
5,21,> 0.4,< 0.03,13,3497,54.303689,1.750959,0.562281
6,13,> 0.4,< 0.08,13,9692,51.124639,1.093996,0.550621
7,21,> 0.4,< 0.08,13,5022,52.548785,1.445535,0.538306
8,34,> 0.5,< 0.08,13,536,55.783582,3.999420,0.516518
9,21,> 0.4,< 0.05,13,4171,52.912971,1.500907,0.512904
